# Model Monitoring with SageMaker Model Monitor
This notebook sets up data capture and monitoring for the deployed endpoint.

## 1. Setup and Configuration

In [8]:
import sagemaker
from sagemaker import get_execution_role
from sagemaker.model_monitor import DataCaptureConfig

role = get_execution_role()
session = sagemaker.Session()
bucket = session.default_bucket()

endpoint_name = "final-model-endpoint-1750546481"  # Make sure this matches your deployed endpoint
print("Using endpoint:", endpoint_name)


Using endpoint: final-model-endpoint-1750546481


## 2. Enable Data Capture on Endpoint

In [9]:
import boto3

sm_client = boto3.client('sagemaker')
endpoints = sm_client.list_endpoints()

for ep in endpoints['Endpoints']:
    print(ep['EndpointName'])


final-model-endpoint-1750546481
xgb-model-endpoint


In [10]:
from sagemaker.model_monitor import ModelMonitor

data_capture_config = DataCaptureConfig(
    enable_capture=True,
    sampling_percentage=100,
    destination_s3_uri=f"s3://{bucket}/data-capture",
    capture_options=["REQUEST", "RESPONSE"]
)

predictor = sagemaker.predictor.Predictor(endpoint_name=endpoint_name, sagemaker_session=session)
predictor.update_data_capture_config(data_capture_config=data_capture_config)

print("Data capture configuration updated.")


------!Data capture configuration updated.


## 3. Schedule a Data Quality Monitoring Job

In [12]:
from sagemaker.model_monitor import DefaultModelMonitor

monitor = DefaultModelMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    volume_size_in_gb=20,
    max_runtime_in_seconds=3600,
    sagemaker_session=session,
)

baseline_dataset = f"s3://{bucket}/final_project/feature_engineer/Xy_train.csv"
baseline_output = f"s3://{bucket}/monitoring/baseline"

monitor.suggest_baseline(
    baseline_dataset=baseline_dataset,
    dataset_format={"csv": {"header": True}},
    output_s3_uri=baseline_output,
    wait=True
)


INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker:Creating processing-job with name baseline-suggestion-job-2025-06-21-23-08-57-869


................2025-06-21 23:11:33.306221: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2025-06-21 23:11:33.306260: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
2025-06-21 23:11:35.037546: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory
2025-06-21 23:11:35.037580: W tensorflow/stream_executor/cuda/cuda_driver.cc:269] failed call to cuInit: UNKNOWN ERROR (303)
2025-06-21 23:11:35.037605: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:156] kernel driver does not appear to be running on this host (ip-10-0-169-140.ec2.internal): /proc/driver/nvidia/version does not exist
2025-06-21 23:11:35.037913: I te

## 4. Create a Monitoring Schedule

In [13]:
from datetime import datetime

monitor_schedule_name = "monitor-schedule-{}".format(datetime.now().strftime("%Y%m%d%H%M"))

monitor.create_monitoring_schedule(
    monitor_schedule_name=monitor_schedule_name,
    endpoint_input=endpoint_name,
    output_s3_uri="s3://{}/monitoring/reports".format(bucket),
    statistics="Enabled",
    constraints="Enabled",
    schedule_cron_expression="cron(0 * ? * * *)"  # Every hour
)

print("Monitoring schedule created:", monitor_schedule_name)


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:5                                                                                    │
│                                                                                                  │
│    2                                                                                             │
│    3 monitor_schedule_name = "monitor-schedule-{}".format(datetime.now().strftime("%Y%m%d%H%M    │
│    4                                                                                             │
│ ❱  5 monitor.create_monitoring_schedule(                                                         │
│    6 │   monitor_schedule_name=monitor_schedule_name,                                            │
│    7 │   endpoint_input=endpoint_name,                                                           │
│    8 │   output_s3_uri="s3://{}/monitoring/reports".format(bucket),                              │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/model_monitor/model_monitoring.py:2020 in      │
│ create_monitoring_schedule                                                                       │
│                                                                                                  │
│   2017 │   │   │   schedule_name=monitor_schedule_name                                           │
│   2018 │   │   )                                                                                 │
│   2019 │   │   new_job_definition_name = name_from_base(self.JOB_DEFINITION_BASE_NAME)           │
│ ❱ 2020 │   │   request_dict = self._build_create_data_quality_job_definition_request(            │
│   2021 │   │   │   monitoring_schedule_name=monitor_schedule_name,                               │
│   2022 │   │   │   job_definition_name=new_job_definition_name,                                  │
│   2023 │   │   │   image_uri=self.image_uri,                                                     │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/model_monitor/model_monitoring.py:2762 in      │
│ _build_create_data_quality_job_definition_request                                                │
│                                                                                                  │
│   2759 │   │                                                                                     │
│   2760 │   │   # baseline config                                                                 │
│   2761 │   │   # noinspection PyTypeChecker                                                      │
│ ❱ 2762 │   │   statistics_object, constraints_object = self._get_baseline_files(                 │
│   2763 │   │   │   statistics=statistics,                                                        │
│   2764 │   │   │   constraints=constraints,                                                      │
│   2765 │   │   │   sagemaker_session=self.sagemaker_session,                                     │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/model_monitor/model_monitoring.py:1292 in      │
│ _get_baseline_files                                                                              │
│                                                                                                  │
│   1289 │   │                                                                                     │
│   1290 │   │   """                                                                               │
│   1291 │   │   if statistics is not None and isinstance(statistics, string_types):               │
│ ❱ 1292 │   │   │   statistics = Statistics.from_s3_uri(    

## 5. Review Reports (Manually in S3 or Use SageMaker Studio)